### **PASO 1: Instalación de Paquetes Necesarios**
Se instalan las bibliotecas necesarias para que el chatbot funcione correctamente.

In [ ]:
import sys
print(sys.version)

# Instalación de bibliotecas necesarias
%pip install transformers
%pip install sentence_transformers
%pip install hnswlib
%pip install numpy<2.0  # Especificar versión de numpy menor que 2.0
%pip install PyPDF2  # Para extraer texto de archivos PDF
%pip install python-dotenv  # Para manejo de variables de entorno desde .env
%pip install tenacity  # Para manejar reintentos en llamadas API
%pip install llama-index  # Para integración con Gemini
%pip install llama-index-llms-gemini  # Para usar Gemini como LLM
%pip install llama-index-embeddings-gemini  # Para embeddings relacionados con Gemini
%pip install tqdm  # mostrar barras de progreso


### **PASO 2: Importar Librerías y Configurar Logging**
Se importan las librerías necesarias y se configura un sistema de logs para monitorear todo el flujo.


In [ ]:
# Importar todas las librerías necesarias
import os
import json
import logging
import hnswlib
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
import numpy as np
from dotenv import load_dotenv
from PyPDF2 import PdfReader
from tenacity import retry, wait_exponential, stop_after_attempt
from llama_index.llms.gemini import Gemini
from llama_index.core.llms import ChatMessage

# Configuración de logging para imprimir todo en consola
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]  # Solo consola
)

logging.info("Librerías importadas correctamente.")  # Mensaje de log para confirmar importación

# Cargar variables de entorno desde un archivo .env
# Assuming your .env file is in 'gdrive/MyDrive/Colab Notebooks/DIA/'
load_dotenv()
logging.info("Variables de entorno cargadas desde el archivo .env.")

### **PASO 3: Cargar Documentos**
Este paso carga documentos desde un archivo o un directorio. Soporta formatos .txt, .json y .pdf.


In [ ]:
def load_documents(source, is_directory=False):
    """
    Carga documentos desde un archivo o un directorio. Soporta .txt, .json y .pdf.
    Además, divide los archivos .txt en unidades usando un delimitador específico.
    """
    loaded_files = []

    # Verificar si la ruta existe
    if not os.path.exists(source):
        logging.error(f"La fuente '{source}' no existe.")
        raise FileNotFoundError(f"La fuente '{source}' no se encontró.")

    if is_directory:
        logging.info(f"Iniciando carga desde el directorio: {source}.")
        for filename in os.listdir(source):
            filepath = os.path.join(source, filename)
            if os.path.isfile(filepath) and filepath.endswith(('.txt', '.json', '.pdf')):
                content = extract_content(filepath)
                if content:
                    loaded_files.append({"filename": filename, "content": content})
                    logging.info(f"Archivo '{filename}' cargado correctamente.")
    else:
        logging.info(f"Iniciando carga del archivo: {source}.")
        content = extract_content(source)
        if content:
            loaded_files.append({"filename": os.path.basename(source), "content": content})
            logging.info(f"Archivo '{os.path.basename(source)}' cargado correctamente.")

    logging.info(f"{len(loaded_files)} documentos cargados.")
    return loaded_files

def extract_content(filepath):
    """
    Extrae el contenido del archivo según su tipo (.txt, .json, .pdf).
    Si el archivo es .txt, lo divide en unidades por el delimitador '\n------\n'.
    """
    try:
        if filepath.endswith('.txt'):
            with open(filepath, 'r', encoding='utf-8') as file:
                content = file.read()
            # Dividir el contenido en unidades por el delimitador
            units = content.split("\n-----\n")
            return units  # Devolver una lista de unidades
        elif filepath.endswith('.json'):
            with open(filepath, 'r', encoding='utf-8') as file:
                return json.load(file)
        elif filepath.endswith('.pdf'):
            reader = PdfReader(filepath)
            return ''.join(page.extract_text() or '' for page in reader.pages)
    except Exception as e:
        logging.error(f"Error al extraer contenido de '{filepath}': {e}")
        return None

# Cargar documentos (True: busca en directorio, False: busca archivo)
ruta_fuente = 'data' # Usando un directorio como ejemplo
documentos = load_documents(ruta_fuente, is_directory=True)

# Visualizar cuántos documentos fueron cargados
logging.info(f"Se cargaron {len(documentos)} documentos exitosamente.")


### **PASO 4: Configurar la Clave API de Gemini**
Configura la conexión con el modelo de lenguaje Gemini usando la clave API.

In [ ]:
gemini_llm = None

def configure_gemini():
    """
    Configura la instancia de Gemini usando la clave API.
    """
    global gemini_llm
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        logging.error("La clave API de Gemini no está configurada.")
        raise EnvironmentError("Configura GEMINI_API_KEY en tu archivo .env.")
    gemini_llm = Gemini(api_key=api_key)
    logging.info("Gemini configurado correctamente.")

# Configurar Gemini
configure_gemini()

### PASO 5
Configurar el modelo de embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer, util
model_name = "all-MiniLM-L6-v2"  # Modelo para embeddings
model = SentenceTransformer(model_name)

def doc_enfermedad(pregunta):
  """
  Identifica el archivo de donde leera la información sobre la enfermedad en la pregunta,
  buscando el mayor embedding entre ella y los nombres de los archivos en documentos
  """
  preg_embedding=model.encode(pregunta)
  archivos = [documentos[i]['filename'] for i in range(0,len(documentos),1)]
  doc_filenames_embeddings = [model.encode(name) for name in archivos]
  similarities = [util.cos_sim(preg_embedding, doc_emb).item() for doc_emb in doc_filenames_embeddings]
  #identificar el índice de la categoría más similar
  max_index = similarities.index(max(similarities))
  max_similarity = similarities[max_index]
  lugar=0
  for i in range(0,len(documentos),1):
    if archivos[i]==max_index:
      lugar=i
  return max_index

### PASO 6
Crear clases para documentos e índices.

In [ ]:
# Clase para documentos
class Document:
    def __init__(self, text, metadata=None):
        self.page_content = text
        self.metadata = metadata or {}
    
    def __str__(self):
        # Acceder a los metadatos de forma segura utilizando .get()
        return (
            f"Título: {self.metadata.get('Title', 'N/A')}\n"
            f"Resumen: {self.metadata.get('Summary', 'N/A')}\n"
            f"Tipo de Estudio: {self.metadata.get('StudyType', 'N/A')}\n"
            f"Paises donde se desarrolla el estudio: {self.metadata.get('Countries', 'N/A')}\n"
            f"Fase en que se encuentra el estudio: {self.metadata.get('Phases', 'N/A')}\n"
            f"Identificación en ClinicaTrial: {self.metadata.get('IDestudio', 'N/A')}.\n\n"
        )

# Clase para HNSWlibIndex
class HNSWIndex:
    def __init__(self, embeddings, metadata=None, space='cosine', ef_construction=200, M=16):
        self.dimension = embeddings.shape[1]
        self.index = hnswlib.Index(space=space, dim=self.dimension)
        self.index.init_index(max_elements=embeddings.shape[0], ef_construction=ef_construction, M=M)
        self.index.add_items(embeddings, np.arange(embeddings.shape[0]))
        self.index.set_ef(50)  # Parámetro ef para consultas
        self.metadata = metadata or []
    
    def similarity_search(self, query_vector, k=5):
        labels, distances = self.index.knn_query(query_vector, k=k)
        return [(self.metadata[i], distances[0][j]) for j, i in enumerate(labels[0])]


### PASO 7
Se procesan los documentos y se crea un índice HNSWlib para cada conjunto de documentos.

In [ ]:
# Procesar documentos y crear índice
def desdobla_doc(data2):
    documents = []
    summaries = []

    for entry in data2['content']:
        nctId = entry.get("IDestudio", "")
        briefTitle = entry.get("Title", "")
        summary = entry.get("Summary", "")
        studyType = entry.get("StudyType", "")
        country = entry.get("Countries", "")
        overallStatus = entry.get("OverallStatus", "")
        conditions = entry.get("Conditions", "")
        phases = entry.get("Phasess", "")

        Summary = (
            f"The study titled '{briefTitle}', of type '{studyType}', "
            f"is being conducted to investigate the condition(s) {conditions}. "
            f"This study is briefly summarized as follows: {summary}. "
            f"Currently, the study status is {overallStatus}, and it is taking place in {country}. "
            f"The study is classified under {phases} phase. "
            f"For more information, search {nctId} on ClinicalTrials."
        )
        metadata = {
            "Summary": Summary
        }
        documents.append(Document(Summary, metadata))
        summaries.append(Summary)

    # Crear embeddings y HNSWIndex una sola vez
    embeddings = model.encode([doc.page_content for doc in documents], show_progress_bar=True)
    embeddings = np.array(embeddings).astype(np.float32)  # Asegurarse de que sean float32
    vector_store = HNSWIndex(embeddings, metadata=[doc.metadata for doc in documents])
    return documents, vector_store

# Genera trozos e índices para las enfermedades y los guarda
trozos_archivos = []
index_archivos = []
for i in range(len(documentos)):
    trozos, index = desdobla_doc(documentos[i])
    trozos_archivos.append(trozos)
    index_archivos.append(index)

logging.info("Índices HNSWlib creados para todos los documentos.")


### **PASO 8: Traducir Preguntas y Respuestas**
Traduce preguntas y respuestas entre idiomas según sea necesario utilizando Gemini.

In [ ]:
import time
def traducir(texto, idioma_destino):
    """
    Traduce texto al idioma especificado.
    """
    start_time = time.time()  # Medir el tiempo de traducción

    mensajes = [
        ChatMessage(role="system", content="Actúa como un traductor."),
        ChatMessage(role="user", content=f"Por favor, traduce este texto al {idioma_destino}: {texto}")
    ]
    try:
        respuesta = gemini_llm.chat(mensajes)
        elapsed_time = time.time() - start_time  # Tiempo de traducción
        logging.info(f"Traducción completada: {respuesta.message.content.strip()} en {elapsed_time:.2f} segundos.")
        return respuesta.message.content.strip()
    except Exception as e:
        logging.error(f"Error al traducir: {e}")
        return texto  # Devolver texto original como fallback

# Función para generar embedding de una pregunta usando TF-IDF
def generate_embedding(texto):
    """
    Genera un embedding TF-IDF para una pregunta de texto.
    """
    try:
        #embedding = tfidf_vectorizer.transform([texto]).toarray()  # Generar embedding con TF-IDF
        embedding = model.encode([texto])
        logging.info(f"Embedding generado para el texto: {texto}")
        return embedding
    except Exception as e:
        logging.error(f"Error al generar el embedding: {e}")
        return np.zeros((1, 300))  # Devolver un embedding vacío en caso de error

def obtener_contexto(pregunta, index, trozos, top_k=50):
    """
    Recupera los trozos de texto más relevantes para responder la pregunta.
    Traduce la pregunta al inglés antes de buscar en el índice.

    Parámetros:
    - pregunta (str): La pregunta del usuario.
    - index (hnswlib.Index): Índice HNSWlib con embeddings.
    - trozos (list): Lista de textos troceados.
    - top_k (int): Número de trozos relevantes a recuperar.

    Retorna:
    - str: Contexto concatenado con los trozos más relevantes.
    """
    try:
        # Traducir la pregunta al inglés
        pregunta_en_ingles = traducir(pregunta, "inglés")
        logging.info(f"Pregunta traducida al inglés: {pregunta_en_ingles}")

        # Generar embedding de la pregunta traducida
        pregunta_emb = generate_embedding(pregunta_en_ingles)
        logging.info("Embedding generado para la pregunta.")

        #Usar indice correspondiente a la enfermedad sobre la que se pregunta
        results = index.similarity_search(pregunta_emb, k=50)
        texto = ""
        for entry in results:
            resum = entry[0]["Summary"]
            texto += resum + "\n"

        # Realizar búsqueda en el índice HNSWlib
        #labels, _ = index.knn_query(pregunta_emb.reshape(1, -1), k=top_k)
        #contexto = ' '.join(trozos[i] for i in labels[0])
        contexto=texto
        logging.info("Contexto relevante recuperado para la pregunta.")
        return contexto
    except Exception as e:
        logging.error(f"Error al obtener el contexto: {e}")
        return ""

### **PASO 9: Generar Respuestas**
Genera respuestas utilizando el modelo Gemini y el contexto proporcionado.

In [ ]:
def generar_respuesta(pregunta, contexto):
    """
    Genera una respuesta usando el contexto proporcionado.
    """
    start_time = time.time()  # Medir el tiempo de generación de respuesta

    mensajes = [
        ChatMessage(role="system", content="Eres un experto médico. "),
        ChatMessage(role="user", content=f"Contexto: {contexto}\nPregunta: {pregunta}")
    ]
    try:
        respuesta = gemini_llm.chat(mensajes)
        elapsed_time = time.time() - start_time  # Tiempo de generación de respuesta
        logging.info(f"Respuesta generada en inglés: {respuesta.message.content.strip()} en {elapsed_time:.2f} segundos.")

        # Traducir la respuesta al español
        respuesta_en_espanol = traducir(respuesta.message.content, "español")
        logging.info(f"Respuesta traducida al español: {respuesta_en_espanol}")
        return respuesta_en_espanol
    except Exception as e:
        logging.error(f"Error al generar la respuesta: {e}")
        return "Lo siento, ocurrió un error al generar la respuesta."

### **PASO 10: Función Principal para Responder Preguntas**
Integra todos los pasos previos para traducir, recuperar contexto y generar respuestas.

In [ ]:
def responder_pregunta(pregunta, index, trozos):
    """
    Integra todos los pasos: traducción, recuperación de contexto y generación de respuestas.
    """
    try:
        # Obtener el contexto relevante para la pregunta
        contexto = obtener_contexto(pregunta, index, trozos)
        if not contexto:
            logging.warning("No se encontró un contexto relevante para la pregunta.")
            return "No pude encontrar información relevante para responder tu pregunta."

        # Generar la respuesta utilizando el contexto
        respuesta = generar_respuesta(pregunta, contexto)
        return respuesta
    except Exception as e:
        logging.error(f"Error en el proceso de responder pregunta: {e}")
        return "Ocurrió un error al procesar tu pregunta."


### **PASO 11: Interfaz CLI**
Proporciona una interfaz interactiva para que los usuarios puedan hacer preguntas.

In [ ]:
if __name__ == "__main__":
    print("Bienvenido al chatbot de ensayos clínicos")
    print("Conversemos sobre Ensayos Clínicos\n de las siguientes enfermedades neuromusculares: Distrofia Muscular de Duchenne o de Becker, Enfermedad de Pompe, Distrofia Miotónica o Enfermedad de almacenamiento de glucosis")
    print("Escribe tu pregunta, dejando claramente expresada la enfermedad sobre la que quieres conocer información de ensayos médicos. Escribe 'salir' para terminar.")
    while True:
        # Solicitar la pregunta al usuario
        pregunta = input("Tu pregunta: ").strip()

        # Verificar si el usuario desea salir
        if pregunta.lower() in ['salir', 'chau', 'exit', 'quit']:
            print("¡Chau!")
            logging.info("El usuario ha finalizado la sesión.")
            break
        #identificar enfermedad
        idn=doc_enfermedad(pregunta)
        index=index_archivos[idn]
        trozos=trozos_archivos[idn]
        # Generar la respuesta para la pregunta del usuario
        respuesta = responder_pregunta(pregunta, index, trozos)
        print(f"Respuesta: {respuesta}")